In [5]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [10]:

print("=" * 60)
print("ÉTAPE 1 : Téléchargement du dataset via KaggleHub")
print("=" * 60)

path = kagglehub.dataset_download("mohammedtawfikmusaed/asthma-detection-dataset-version-2")
print(f"✅ Dataset téléchargé dans : {path}")
print("\n📁 Structure du dataset :")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    if level < 3:  # Limiter l'affichage
        for f in files[:5]:
            print(f"{indent}  📄 {f}")
        if len(files) > 5:
            print(f"{indent}  ... et {len(files)-5} autres fichiers")

ÉTAPE 1 : Téléchargement du dataset via KaggleHub
✅ Dataset téléchargé dans : /Users/eliekanga/.cache/kagglehub/datasets/mohammedtawfikmusaed/asthma-detection-dataset-version-2/versions/1

📁 Structure du dataset :
📂 1/
  📄 Des.txt
  📂 Asthma Detection Dataset Version 2/
    📄 Des.txt
    📂 Asthma Detection Dataset Version 2/
      📂 healthy/
      📂 Bronchial/
      📂 asthma/
      📂 pneumonia/
      📂 copd/


In [11]:
# ============================================================
# 2. CONSTRUCTION DU CATALOGUE DE FICHIERS
# ============================================================

print("\n" + "=" * 60)
print("ÉTAPE 2 : Construction du catalogue de fichiers audio")
print("=" * 60)

# Mapper les noms de dossiers possibles vers les classes standardisées
CLASS_MAPPING = {
    'asthma':    'Asthma',
    'asthme':    'Asthma',
    'copd':      'COPD',
    'bpco':      'COPD',
    'bronchial': 'Bronchial',
    'bronchique':'Bronchial',
    'pneumonia': 'Pneumonia',
    'pneumonie': 'Pneumonia',
    'healthy':   'Healthy',
    'sain':      'Healthy',
    'normal':    'Healthy',
}

records = []

for wav_path in Path(path).rglob("*.wav"):
    # Détecter la classe depuis le nom du dossier parent
    folder = wav_path.parent.name.lower().strip()
    label = CLASS_MAPPING.get(folder, folder.capitalize())
    
    records.append({
        'filepath':  str(wav_path),
        'filename':  wav_path.name,
        'label':     label,
        'folder':    folder,
    })

df = pd.DataFrame(records)
print(f"✅ {len(df)} fichiers audio trouvés")
print(f"📊 Classes détectées : {df['label'].unique().tolist()}")
print(f"\n{df.head(10)}")



ÉTAPE 2 : Construction du catalogue de fichiers audio
✅ 1211 fichiers audio trouvés
📊 Classes détectées : ['Healthy', 'Bronchial', 'Asthma', 'Pneumonia', 'COPD']

                                            filepath           filename  \
0  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P11Healthy30S.wav   
1  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P13Healthy69S.wav   
2  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P24Healthy58S.wav   
3  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P19Healthy62S.wav   
4  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P13Healthy28S.wav   
5  /Users/eliekanga/.cache/kagglehub/datasets/moh...   P6Healthy34S.wav   
6  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P13Healthy53S.wav   
7  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P24Healthy62S.wav   
8  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P25Healthy20S.wav   
9  /Users/eliekanga/.cache/kagglehub/datasets/moh...  P23Healthy14S.wav   

     label